# Galerie GenAI — 02 · Le menu des techniques RAG : un symptôme, un remède, une mesure 🟢→🟠

> ⚠️ **À faire APRÈS ton cas d'usage certif** — et après le notebook 01,
> dont on réutilise le corpus, la baseline et surtout le **harnais d'éval**.
>
> **Étagère optionnelle** — pas un brief, pas de livrable, pas de note.
> **Autonomie** : 🟢 sections 1-4 résolues · 🟠 sections 5-6 **à compléter**
> (`# TODO`, pas de solution).
> **Durée** : ~3 h · les sections sont **indépendantes** — picore.
> **Références** : *RAG made simple* (N. Diamant) — chaque section renvoie à
> son chapitre ; son **Appendix A** (« What Symptom Are You Seeing? ») est
> l'esprit de ce notebook. Code ouvert : `github.com/NirDiamant/RAG_Techniques`.

## La règle du jeu

Ton RAG du notebook 01 marche (hit@1 ≈ 0,87). La tentation maintenant :
empiler des techniques « parce que le repo en liste 30 ». La méthode pro,
c'est l'inverse — la même qu'en M4-B1 :

> **constate un symptôme → applique UNE technique → re-mesure → décide.**

Chaque section suit ce cycle avec le harnais du 01 comme juge de paix. À la
fin, tu repars avec une **grille de décision RAG** remplie de chiffres à toi.

## Setup — les acquis du notebook 01, condensés

(Chargement, chunking par sections, index, `cherche`, harnais d'éval — tout
est expliqué au notebook 01.)

In [ ]:
import json
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import requests

RANDOM_STATE = 42
OLLAMA_URL = "http://localhost:11434"
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "qwen2.5:1.5b")
MOCK_MODE = os.getenv("MOCK_MODE", "0") == "1"

if not MOCK_MODE:
    try:
        requests.get(f"{OLLAMA_URL}/api/tags", timeout=3)
        print(f"✅ Ollama joignable, modèle : {OLLAMA_MODEL}")
    except Exception:
        MOCK_MODE = True
        print("⚠️ Ollama injoignable → MOCK_MODE (retrieval réel ; rewriting, HyDE et juge non représentatifs).")


def ollama(consigne_systeme: str, message: str, max_tokens: int = 150) -> str:
    reponse = requests.post(f"{OLLAMA_URL}/api/chat", timeout=180, json={
        "model": OLLAMA_MODEL, "stream": False,
        "options": {"temperature": 0, "num_predict": max_tokens},
        "messages": [{"role": "system", "content": consigne_systeme},
                     {"role": "user", "content": message}]})
    reponse.raise_for_status()
    return reponse.json()["message"]["content"].strip()

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb

modele_embedding = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
documents = {p.name: p.read_text(encoding="utf-8")
             for p in sorted(Path("corpus_fastia").glob("*.md"))}


def chunks_par_sections(nom_doc, texte):
    titre_doc = texte.splitlines()[0].lstrip("# ").strip()
    chunks = []
    for section in texte.split("\n## ")[1:]:
        titre_section, _, corps = section.partition("\n")
        chunks.append({"texte": f"{titre_doc} > {titre_section.strip()}\n{corps.strip()}",
                       "source": nom_doc})
    return chunks


corpus = [c for nom, txt in documents.items() for c in chunks_par_sections(nom, txt)]
client = chromadb.EphemeralClient()
index_sections = client.create_collection("sections", metadata={"hnsw:space": "cosine"})
vecteurs = modele_embedding.encode([c["texte"] for c in corpus],
                                   normalize_embeddings=True, show_progress_bar=False)
index_sections.add(ids=[str(i) for i in range(len(corpus))], embeddings=vecteurs.tolist(),
                   documents=[c["texte"] for c in corpus],
                   metadatas=[{"source": c["source"]} for c in corpus])


def cherche(question, collection=index_sections, k=3):
    v = modele_embedding.encode([question], normalize_embeddings=True)
    r = collection.query(query_embeddings=v.tolist(), n_results=k)
    return [{"texte": d, "source": m["source"], "similarite": round(1 - dist, 3)}
            for d, m, dist in zip(r["documents"][0], r["metadatas"][0], r["distances"][0])]


jeu_eval = json.loads(Path("eval/questions_eval.json").read_text(encoding="utf-8"))
questions_eval = jeu_eval["questions"]


def mesure(retrieveur, questions=questions_eval, k=3) -> dict:
    """Le juge de paix : hit@1 / hit@3 d'un retrieveur sur un jeu de questions."""
    h1 = h3 = 0
    for q in questions:
        sources = [r["source"] for r in retrieveur(q["question"], k)]
        h1 += sources[0] == q["doc_attendu"]
        h3 += q["doc_attendu"] in sources
    return {"hit@1": round(h1 / len(questions), 2), "hit@3": round(h3 / len(questions), 2)}


retrieveur_base = lambda q, k=3: cherche(q, index_sections, k)
print("BASELINE (notebook 01) :", mesure(retrieveur_base))

---

## §1 · Symptôme : « vos utilisateurs ne parlent pas comme vos documents »

> 📖 *RAG made simple* ch. 5 (Query Transformations) ·
> repo : `query-transformations`

Ton jeu d'éval du 01 est écrit en bon français documentaire. Les vrais
utilisateurs, eux, écrivent : *« ça se passe comment les ndf ? »*. Mesurons
d'abord les dégâts sur 8 questions **dégradées** (oral + jargon interne) :

In [ ]:
questions_degradees = [
    {"question": "c'est quoi le max de jours en tt par semaine ?", "doc_attendu": "politique_teletravail.md"},
    {"question": "du coup on a droit à combien pour un resto le midi en dép ?", "doc_attendu": "procedure_notes_de_frais.md"},
    {"question": "jusqu'à quand je peux poser mes vieux congés ?", "doc_attendu": "faq_conges.md"},
    {"question": "le truc pour le vpn là, il dure combien de temps ?", "doc_attendu": "procedure_acces_vpn.md"},
    {"question": "faut un mdp de combien de caractères déjà ?", "doc_attendu": "charte_securite_poste.md"},
    {"question": "on touche combien quand on est d'astreinte ?", "doc_attendu": "procedure_astreinte.md"},
    {"question": "ça se passe comment les ndf ?", "doc_attendu": "procedure_notes_de_frais.md"},
    {"question": "c'est quand la recette meridian ?", "doc_attendu": "cr_projet_meridian_avril.md"},
]

print("questions propres    :", mesure(retrieveur_base))
print("questions dégradées  :", mesure(retrieveur_base, questions_degradees))

Le hit@1 s'effondre. Deux remèdes, du plus sobre au plus sophistiqué.

### Remède A — la normalisation par règles (15 lignes, zéro latence)

Le jargon interne d'une boîte est **fini et connu** : un glossaire
d'abréviations + une liste de tournures orales à retirer. C'est bête,
déterministe, gratuit — et ça se maintient dans un fichier de config.

In [ ]:
GLOSSAIRE = {" tt ": " télétravail ", "mdp": "mot de passe",
             "ndf": "notes de frais", " dép ": " déplacement ", " dép?": " déplacement"}
TOURNURES_ORALES = ["c'est quoi", "du coup", "déjà", "le truc pour", "là,",
                    "ça se passe comment"]


def normalise(question: str) -> str:
    q = " " + question.lower() + " "
    for abrege, complet in GLOSSAIRE.items():
        q = q.replace(abrege, complet)
    for tournure in TOURNURES_ORALES:
        q = q.replace(tournure, " ")
    return re.sub(r"\s+", " ", q).strip()


retrieveur_normalise = lambda q, k=3: cherche(normalise(q), index_sections, k)
print("dégradées + normalisation :", mesure(retrieveur_normalise, questions_degradees))
print("exemple :", questions_degradees[6]["question"], "→", normalise(questions_degradees[6]["question"]))

### Remède B — le rewriting par LLM (ch. 5)

Le LLM réécrit la question en requête documentaire. Plus général que les
règles… **si le modèle est à la hauteur**. Regarde bien les réécritures.

In [ ]:
CONSIGNE_REWRITE = (
    "Tu réécris des questions orales en requêtes documentaires précises pour la "
    "base RH/IT de FastIA. Glossaire interne : tt = télétravail ; ndf = notes de "
    "frais ; mdp = mot de passe ; dép = déplacement professionnel. Retire les "
    "tournures orales, garde le sens exact, n'ajoute AUCUNE information. "
    "Réponds UNIQUEMENT avec la question réécrite."
)


def rewrite_llm(question: str) -> str:
    if MOCK_MODE:
        return normalise(question)  # en MOCK, le LLM est simulé par les règles
    return ollama(CONSIGNE_REWRITE, question, 60)


for q in questions_degradees[:4]:
    print(q["question"], "\n   →", rewrite_llm(q["question"]), "\n")

retrieveur_rewrite = lambda q, k=3: cherche(rewrite_llm(q), index_sections, k)
bilan_s1 = pd.DataFrame({
    "sans rien": mesure(retrieveur_base, questions_degradees),
    "règles + glossaire": mesure(retrieveur_normalise, questions_degradees),
    "rewriting LLM": mesure(retrieveur_rewrite, questions_degradees),
})
bilan_s1

**Lecture honnête** (nos mesures avec un modèle 3B) : les règles remontent le
hit@1 de 0,50 à ~0,88 ; le rewriting LLM, lui, **plafonne autour de 0,50** —
un petit modèle ne connaît pas ton jargon, il « réécrit » *ndf* en demandant
ce que ça veut dire, ou invente un autre sens à *tt*. Même avec le glossaire
dans le prompt, il reste en dessous des règles. Avec un modèle 7B+, l'écart
se resserre — **mesure avec le tien** avant de trancher.

> 🧭 Verdict type : règles d'abord (sobres, prévisibles, maintenables), LLM
> en complément seulement si tes mesures le justifient. Le réflexe
> « commence simple » du parcours s'applique aussi au RAG.

---

## §2 · Symptôme : « la question est trop courte pour matcher » — HyDE

> 📖 ch. 6 (Hypothetical Document Embedding) · repo : `HyDE`

Idée élégante : plutôt que de chercher avec la **question**, demander au LLM
d'écrire un **document hypothétique** qui y répondrait, et chercher avec ce
document (qui ressemble davantage aux vrais documents). Testons — sur notre
jeu de référence, pas sur la promesse de l'article.

In [ ]:
CONSIGNE_HYDE = (
    "Rédige un court extrait (2-3 phrases) d'un document interne d'entreprise "
    "fictif qui répondrait à la question. Style procédure RH/IT. Invente des "
    "valeurs plausibles."
)


def document_hypothetique(question: str) -> str:
    if MOCK_MODE:
        return (f"Document interne. {question} La procédure précise les modalités, "
                "les délais et les montants applicables aux salariés.")
    return ollama(CONSIGNE_HYDE, question, 120)


def retrieveur_hyde(question, k=3):
    return cherche(document_hypothetique(question), index_sections, k)


print("exemple de document hypothétique :\n",
      document_hypothetique(questions_eval[0]["question"])[:200])
bilan_s2 = pd.DataFrame({"baseline": mesure(retrieveur_base),
                         "HyDE": mesure(retrieveur_hyde)})
bilan_s2

**Lecture honnête** (mesures 3B) : ici HyDE ne gagne **rien** — il recule
même légèrement en hit@1 (0,80 vs 0,87), pour un coût d'**une génération LLM
par requête** (latence ×5 à ×20). Normal : nos questions d'éval sont déjà
précises et nos documents courts. HyDE paie quand les questions sont très
courtes/vagues face à des documents longs et denses — pas ici.

> 🧭 C'est le point le plus important du notebook : **une technique célèbre
> peut être une mauvaise idée sur TON corpus**. Sans harnais d'éval, tu
> l'aurais adoptée sur sa réputation. (En MOCK_MODE, cette section montre la
> mécanique mais le gabarit ne permet pas de conclure — mesure avec Ollama.)

---

## §3 · Symptôme : « le bon document est dans le top 10, pas dans le top 3 » — reranking

> 📖 ch. 13 (Reranking) · repo : `reranking`

La recherche vectorielle est rapide mais approximative. Le **cross-encoder**,
lui, lit la question ET le chunk **ensemble** — précis mais lent. La
stratégie classique : ratisser large (top 10 dense), re-classer fin
(cross-encoder), garder le top 3. (~470 Mo au premier téléchargement,
multilingue, CPU.)

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/mmarco-mMiniLMv2-L12-H384-v1")


def retrieveur_rerank(question, k=3, k_large=10):
    candidats = cherche(question, index_sections, k_large)
    scores = reranker.predict([(question, c["texte"]) for c in candidats])
    meilleurs = np.argsort(scores)[::-1][:k]
    return [candidats[i] for i in meilleurs]


bilan_s3 = pd.DataFrame({"baseline": mesure(retrieveur_base),
                         "rerank top10→3": mesure(retrieveur_rerank)})
bilan_s3

**Nos mesures** : hit@1 0,87 → 0,93 et hit@3 → **1,00** — le gain le plus
net du menu, et il répare notamment la question « mots de passe » que la
baseline ratait. Le prix : un deuxième modèle à héberger et ~10 paires à
scorer par requête (quelques dizaines de ms sur CPU ici, à re-mesurer sur
tes volumes).

---

## §4 · Symptôme : « mes chunks coupent au milieu des idées » — semantic chunking

> 📖 ch. 9 (Semantic Chunking), ch. 4 (Proposition Chunking) ·
> repo : `semantic-chunking`

Au 01, le chunking **par sections** gagnait parce que nos documents sont
bien structurés. Que vaut un découpage **sémantique automatique** (on coupe
quand le sens change, mesuré par les embeddings) face à cette structure
humaine ? Version minimale en ~20 lignes :

In [ ]:
def chunks_semantiques(nom_doc, texte, seuil=0.55):
    titre = texte.splitlines()[0].lstrip("# ").strip()
    paragraphes = [p.strip() for p in re.split(r"\n\s*\n", texte) if len(p.strip()) > 40]
    if not paragraphes:
        return []
    vecteurs_p = modele_embedding.encode(paragraphes, normalize_embeddings=True,
                                         show_progress_bar=False)
    blocs, bloc_courant, vecteur_courant = [], [paragraphes[0]], vecteurs_p[0]
    for para, vect in zip(paragraphes[1:], vecteurs_p[1:]):
        if float(vecteur_courant @ vect) > seuil:      # même sujet → on continue le bloc
            bloc_courant.append(para)
            somme = vecteur_courant + vect
            vecteur_courant = somme / np.linalg.norm(somme)
        else:                                          # rupture de sens → nouveau bloc
            blocs.append("\n".join(bloc_courant))
            bloc_courant, vecteur_courant = [para], vect
    blocs.append("\n".join(bloc_courant))
    return [{"texte": f"{titre}\n{b}", "source": nom_doc} for b in blocs]


corpus_semantique = [c for nom, txt in documents.items()
                     for c in chunks_semantiques(nom, txt)]
index_semantique = client.create_collection("semantique", metadata={"hnsw:space": "cosine"})
vecteurs_sem = modele_embedding.encode([c["texte"] for c in corpus_semantique],
                                       normalize_embeddings=True, show_progress_bar=False)
index_semantique.add(ids=[str(i) for i in range(len(corpus_semantique))],
                     embeddings=vecteurs_sem.tolist(),
                     documents=[c["texte"] for c in corpus_semantique],
                     metadatas=[{"source": c["source"]} for c in corpus_semantique])

retrieveur_semantique = lambda q, k=3: cherche(q, index_semantique, k)
bilan_s4 = pd.DataFrame({
    f"sections ({len(corpus)} chunks)": mesure(retrieveur_base),
    f"sémantique ({len(corpus_semantique)} chunks)": mesure(retrieveur_semantique),
})
bilan_s4

**Nos mesures** : le sémantique perd en hit@1 (0,80 vs 0,87) mais gagne en
hit@3 (1,00 vs 0,93). Lecture : la **structure humaine** de nos documents
(titres, sections) est une information que l'automatique ne fait que
réapproximer. Le semantic chunking prend sa vraie valeur sur du texte **sans
structure** : mails, transcriptions de réunions, PDF océrisés.

> 🧭 Corollaire d'intégrateur : si tu peux obtenir des documents structurés
> à la source (gabarits, conventions de rédaction), c'est la technique RAG
> la plus rentable de toutes — et elle ne coûte pas une ligne de code.

---

## 🎯 §5 · Symptôme : « les sigles et références exactes se perdent » — fusion BM25 + dense (🟠 à toi)

> 📖 ch. 12 (Fusion Retrieval) · repo : `fusion-retrieval`

L'embedding comprend le **sens**, mais « Notilus », « poste 4242 », « P1 »
n'ont pas de sens — ce sont des **jetons exacts**, le territoire de la
recherche lexicale (BM25). Chacun voit ce que l'autre rate → on fusionne.

D'abord, constate — y compris le **piège des mots-outils** qui plombe BM25 :

In [ ]:
from rank_bm25 import BM25Okapi

questions_vocabulaire = [
    {"question": "À quoi sert Notilus ?", "doc_attendu": "procedure_notes_de_frais.md"},
    {"question": "Qui répond au poste 4242 ?", "doc_attendu": "procedure_acces_vpn.md"},
    {"question": "KeePass est-il obligatoire ?", "doc_attendu": "charte_securite_poste.md"},
    {"question": "Que faire en cas de P1 ?", "doc_attendu": "procedure_incident_production.md"},
]

# Piège : tokenisation naïve → les mots de la question polluent le score
tokenise_naif = lambda t: re.findall(r"\w+", t.lower())
bm25_naif = BM25Okapi([tokenise_naif(c["texte"]) for c in corpus])
scores = bm25_naif.get_scores(tokenise_naif("À quoi sert Notilus ?"))
premier = int(np.argsort(scores)[::-1][0])
print("BM25 naïf, 1er résultat pour 'À quoi sert Notilus ?' :",
      corpus[premier]["source"], "→ piégé par 'quoi' (cf. section 'Qui paie quoi ?')")

# Remède : retirer les mots-outils de la requête ET de l'index
MOTS_OUTILS = set(("le la les un une des de du au aux et ou est sont a ont pour dans sur "
                   "avec sans que qui quoi quand comment ce cette ces son sa ses il elle on "
                   "nous vous ils à en par se faire fait faut sert d l qu s je tu mon ma mes").split())
tokenise = lambda t: [m for m in re.findall(r"\w+", t.lower()) if m not in MOTS_OUTILS]
bm25 = BM25Okapi([tokenise(c["texte"]) for c in corpus])


def cherche_bm25(question, k=3):
    scores = bm25.get_scores(tokenise(question))
    meilleurs = np.argsort(scores)[::-1][:k]
    return [{"texte": corpus[i]["texte"], "source": corpus[i]["source"]} for i in meilleurs]


pd.DataFrame({
    "dense (baseline)": mesure(retrieveur_base, questions_vocabulaire),
    "BM25 (sans mots-outils)": mesure(cherche_bm25, questions_vocabulaire),
})

Sur le vocabulaire exact, le lexical écrase le dense (qui fait ici 0,00 en
hit@1). À toi de les **fusionner** avec la méthode RRF (*Reciprocal Rank
Fusion*) : chaque candidat gagne `1/(k_rrf + rang + 1)` points dans chaque
liste où il apparaît, et on trie par score total. Simple, sans réglage fin,
et étonnamment robuste.

In [ ]:
def cherche_fusion(question, k=3, k_rrf=60):
    candidats_dense = cherche(question, index_sections, k=10)
    candidats_bm25 = cherche_bm25(question, k=10)
    # TODO — fusion RRF :
    #  1) identifie chaque candidat par une clé stable, ex. (source, texte[:40]),
    #     pour reconnaître le même chunk dans les deux listes ;
    #  2) score(candidat) = somme de 1 / (k_rrf + rang + 1) sur les listes où il figure ;
    #  3) retourne les k meilleurs au format habituel [{"texte":…, "source":…}].
    ...


assert cherche_fusion("télétravail", 3) is not None, (
    "cherche_fusion renvoie None : complète le TODO (fusion RRF) avant de mesurer.")

bilan_s5 = pd.DataFrame({
    "dense": mesure(retrieveur_base),
    "BM25": mesure(cherche_bm25),
    "fusion RRF": mesure(cherche_fusion),
})
bilan_s5

### 🧭 Repères pour t'auto-vérifier (§5)

Sur le **jeu de référence de 15 questions**, ta fusion doit faire **au moins
aussi bien que le meilleur des deux** retrieveurs seuls — chez nous elle
atteint le sans-faute (1,00 / 1,00) : les erreurs du dense et celles du
BM25 ne tombent pas aux mêmes endroits, et la fusion encaisse les deux.
Nuance à vérifier toi-même : sur les 4 questions « vocabulaire », la fusion
ne bat PAS BM25 seul (le dense y est si mauvais qu'il dilue) — la fusion
n'est pas magique, c'est une **moyenne de forces**, à mesurer sur TON trafic
réel.

---

## 🎯 §6 · Symptôme : « la réponse a l'air sûre… et elle est fausse » — le juge d'ancrage (🟠 à toi)

> 📖 ch. 3 (Reliable RAG) · repo : `reliable-RAG`, `self-RAG`

Au notebook 01, tu as posé deux lignes de défense (consigne d'abstention,
seuil de similarité). La troisième vérifie **après génération** : un juge
relit la réponse face aux extraits et tranche — ANCRÉE ou NON_ANCRÉE. En cas
de NON_ANCRÉE : on ne montre pas la réponse, on abstient ou on route vers un
humain.

Trois cas de test, dont une hallucination fabriquée exprès :

In [ ]:
extraits_vpn = cherche("Quelle est la durée de validité du certificat VPN ?", index_sections)
bloc_vpn = "\n\n".join(e["texte"] for e in extraits_vpn)

cas_de_test = [
    ("bonne réponse ancrée", "Le certificat VPN est valable 12 mois.", "ANCRÉE"),
    ("hallucination plausible", "Le certificat VPN est valable 24 mois et se renouvelle auprès du manager.", "NON_ANCRÉE"),
    ("hors-sujet confiant", "La prime de cooptation est de 500 € par recrutement.", "NON_ANCRÉE"),
]


def juge_lexical(reponse: str, extraits_texte: str, seuil: float = 0.7) -> str:
    """Juge de secours (MOCK) : part des mots significatifs de la réponse présents dans les extraits."""
    mots = [m for m in re.findall(r"\w+", reponse.lower()) if len(m) > 3]
    presents = sum(m in extraits_texte.lower() for m in mots)
    return "ANCRÉE" if mots and presents / len(mots) >= seuil else "NON_ANCRÉE"

In [ ]:
# TODO — écris la consigne système du juge LLM. Les 3 ingrédients qui comptent :
#  1) ANCRÉE si l'information principale figure dans les extraits, MÊME reformulée ;
#  2) NON_ANCRÉE seulement si une information est absente des extraits ou les contredit ;
#  3) exiger UN SEUL mot de réponse.
# (Un juge mal calibré rejette les bonnes réponses — notre premier essai le faisait.)
CONSIGNE_JUGE = ...


def juge_ancrage(reponse: str, extraits_texte: str) -> str:
    if MOCK_MODE:
        return juge_lexical(reponse, extraits_texte)
    return ollama(CONSIGNE_JUGE, f"Extraits :\n{extraits_texte}\n\nRéponse à vérifier : {reponse}", 15)


for nom, reponse, attendu in cas_de_test:
    verdict = juge_ancrage(reponse, bloc_vpn)
    statut = "✅" if attendu in verdict.upper().replace("É", "E") or attendu in verdict.upper() else "❌"
    print(f"{statut} {nom:26s} attendu={attendu:11s} obtenu={verdict[:25]}")

### 🧭 Repères pour t'auto-vérifier (§6)

Ton juge doit classer les 3 cas correctement. Deux pièges de calibration
vécus pendant la conception de ce notebook :

- une consigne trop stricte (« CHAQUE affirmation explicitement soutenue »)
  fait rejeter les **bonnes** réponses reformulées par un petit modèle ;
- une consigne trop lâche laisse passer l'hallucination « 24 mois ».

Et garde la hiérarchie en tête : un juge trop sévère coûte des réponses
légitimes (rattrapables par un humain), un juge laxiste coûte des **fausses
informations montrées avec assurance** — en documentaire interne, la
sévérité est le bon défaut.

---

## 📋 La grille de décision RAG — ton livrable

Recopie-la dans tes notes avec **tes** chiffres (colonne « gain mesuré ») —
c'est le pendant GenAI de ta grille C4, et l'esprit de l'**Appendix A** du
livre :

| Symptôme observé | Technique | Gain mesuré ici (hit@1) | Coût ajouté | Livre |
|---|---|---|---|---|
| Questions orales / jargon | Normalisation par règles | 0,50 → 0,88 (dégradées) | ~0 (un dict à maintenir) | ch. 5 |
| Idem, si modèle costaud | Rewriting LLM | 0,50 → ~0,50 avec un 3B ⚠️ | +1 génération/requête | ch. 5 |
| Questions courtes vs docs longs | HyDE | 0,87 → 0,80 ici (pas notre cas) | +1 génération/requête | ch. 6 |
| Bon doc dans le top 10, pas top 3 | Reranking cross-encoder | 0,87 → 0,93 (hit@3 = 1,00) | 2ᵉ modèle, +latence | ch. 13 |
| Texte sans structure | Semantic chunking | 0,87 → 0,80 ici (docs déjà structurés) | embeddings à l'indexation | ch. 9 |
| Sigles, codes, références | Fusion BM25 + dense (RRF) | 0,87 → 1,00 (jeu global) | index lexical en plus | ch. 12 |
| Réponses non ancrées | Juge d'ancrage | qualitatif (3/3 cas) | +1 génération/réponse | ch. 3 |

Trois lignes de cette grille disent « **ne pas adopter ici** » — c'est
exactement ce qu'une grille de décision doit savoir dire.

## 🔎 Ce que tu viens de pratiquer

- Le cycle **symptôme → technique → re-mesure** avec un harnais d'éval figé —
  la seule protection contre l'empilement de techniques à la mode.
- Deux techniques qui paient ici (reranking, fusion), deux qui ne paient pas
  (HyDE, semantic chunking) — et pourquoi ça dépend du **corpus**, pas de la
  technique.
- Le rewriting : règles sobres d'abord, LLM sous conditions mesurées.
- Le juge d'ancrage : la 3ᵉ ligne de défense, et sa calibration.

## ⭐ Pour aller plus loin (optionnel)

- **Compose une pile** (Appendix A « Common Technique Stacks ») :
  normalisation → fusion → rerank → génération → juge, et mesure la chaîne
  complète de bout en bout (latence comprise).
- Ch. 2 (*RAG for Structured Data*) : ton corpus a des **tableaux** (plafonds
  de frais, sévérités P1-P4) — que donne une extraction des tableaux vers un
  format requêtable, en complément du RAG textuel ? Très « intégrateur ».
- Ch. 7 (*Contextual Chunk Headers*) : nos chunks-sections ont déjà un
  en-tête `document > section` — enrichis-le (type de doc, date) et mesure.
- Ajoute tes propres questions dégradées à partir de vraies questions
  d'utilisateurs : c'est le meilleur investissement de tout le menu.